# Unified Coordinate-Policy ETNN

**TopoBench TDL Challenge 2026, Track 2**

**Team:** E(n)igma

**Model family:** E(n)-Equivariant Topological Neural Network (ETNN)

This executable report compares the three coordinate semantics exposed
by the consolidated TopoBench implementation:

1. `none`: topological feature message passing without coordinates;
2. `structural_lappe`: fixed graph-derived pseudo-coordinates;
3. `physical`: physical cell geometry and optional E(n)-equivariant
   rank-0 coordinate updates.

GraphUniverse evaluates the first two policies because it provides
combinatorial graph structure but no physical Euclidean coordinates.
The physical policy is validated separately on QM9, where atom positions
have physical meaning. The QM9 analysis combines a paper-faithful native
reference, a protocol- and architecture-matched TopoBench ETNN port, and
a reduced end-to-end integration check of the submitted physical policy.
No physical GraphUniverse result is fabricated.

The report is generated from two committed GraphUniverse `results.json`
files and one compact QM9 comparison artifact. QM9 results remain separate
from the challenge leaderboard results.


## Questions Answered

- Does each coordinate policy have a clear and reproducible data contract?
- Does structural LapPE consistently improve the two GraphUniverse tasks?
- How does the policy effect vary with homophily, degree, and power-law
  regime?
- Do real physical coordinates survive TopoBench lifting and support
  invariant ETNN messages and equivariant coordinate updates end to end?
- Does an architecture-matched TopoBench ETNN core recover native NSAPH
  behavior when molecular data and training protocol are held fixed?

Final model performance is not a minimum criterion for the challenge.
The primary goals are architectural correctness, faithfulness,
TopoBench compatibility, documentation, and robust tests.


## Coordinate Policies and Notation

For a typed neighborhood relation $N$, let $d$ denote a sender cell,
$c$ a receiver cell, $h_d$ and $h_c$ their hidden features, and
$a_{d,c,N}$ the scalar value stored by the sparse TopoBench relation.
A relation-specific gated message is

$$
\widetilde m_{d,c,N}
= \psi_N\!\left([h_d, h_c, e_{d,c,N}]\right),
\qquad
g_{d,c,N}
= \sigma\!\left(w_N^\top \widetilde m_{d,c,N}+b_N\right),
$$

$$
m_{c,N}
= \sum_{d\in N(c)} g_{d,c,N}\,\widetilde m_{d,c,N}.
$$

Here $\psi_N$ is the relation-specific message MLP, $\sigma$ is the
sigmoid function, and $e_{d,c,N}$ depends on the coordinate policy.

| Policy | Relation attribute $e_{d,c,N}$ | Coordinate behavior |
|---|---|---|
| `none` | $a_{d,c,N}$ | No coordinate tensor is required or invented. |
| `structural_lappe` | $[a_{d,c,N},\lVert p_d-p_c\rVert_2^2]$ | Fixed structural pseudo-coordinates; no coordinate update. |
| `physical` | Physical centroid/diameter/Hausdorff invariants | Optional learned E(n)-equivariant rank-0 update. |

For structural LapPE, $p_0(v)$ is the selected normalized graph
Laplacian eigenvector embedding of rank-0 cell $v$. Higher-rank
coordinates are recursively lifted through incidence:

$$
p_r(c)=\frac{\sum_d |B_r(d,c)|\,p_{r-1}(d)}
{\sum_d |B_r(d,c)|},
$$

where $B_r$ is the incidence matrix from rank $r-1$ to rank $r$ and
$d$ ranges over incident lower-rank cells. Squared distances are
invariant to translations and orthogonal transformations of this chosen
structural frame. They do not imply that GraphUniverse has an underlying
physical E(n) action.

Physical mode reconstructs each cell's incident rank-0 vertices, then
computes centroids, diameters, and optional directed Hausdorff-style
distances from current physical positions. When `pos_update=true`, the
rank-0 coordinates are updated radially and all physical invariants are
recomputed before the next ETNN layer. This is the policy closest to the
original ETNN/NSAPH physical-coordinate implementation.

## Setup and Artifact Validation

In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any, Final, Literal, TypedDict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from numpy.typing import NDArray
from scipy import stats

PolicyKey = Literal["none", "lappe"]


class ExperimentSpec(TypedDict):
    """Describe one GraphUniverse task metric and its direction."""

    metric: str
    label: str
    higher_is_better: bool


plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 220,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "font.size": 10,
    }
)


def find_repo_root(start: Path) -> Path:
    """Find the TopoBench repository containing the public artifacts.

    Parameters
    ----------
    start : Path
        Directory from which to search upward.

    Returns
    -------
    Path
        Repository root containing both ``topobench`` and the challenge
        artifact directory.

    Raises
    ------
    FileNotFoundError
        If no valid repository root exists in the ancestor chain.
    """
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "2026_tdl_challenge").is_dir() and (
            candidate / "topobench"
        ).is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the TopoBench repository root.")


ROOT: Final[Path] = find_repo_root(Path.cwd())
RESULT_PATHS: Final[dict[PolicyKey, Path]] = {
    "none": (
        ROOT
        / "2026_tdl_challenge/outputs/etnn_coordinate_policy_none/results.json"
    ),
    "lappe": (
        ROOT
        / "2026_tdl_challenge/outputs/etnn_coordinate_policy_lappe/results.json"
    ),
}
ASSET_DIR: Final[Path] = (
    ROOT / "2026_tdl_challenge/submissions/assets/etnn_coordinate_policy"
)
ASSET_DIR.mkdir(parents=True, exist_ok=True)

POLICY_LABELS: Final[dict[PolicyKey, str]] = {
    "none": "No coordinates",
    "lappe": "Structural LapPE",
}
POLICY_COLORS: Final[dict[PolicyKey, str]] = {
    "none": "#2F5D62",
    "lappe": "#C65D3B",
}
HOMOPHILY_ORDER: Final[tuple[str, ...]] = ("h_lo", "h_mid", "h_hi")
DEGREE_ORDER: Final[tuple[str, ...]] = ("d_lo", "d_hi")
POWER_LAW_ORDER: Final[tuple[str, ...]] = ("pl_lo", "pl_hi")
EXPERIMENTS: Final[dict[str, ExperimentSpec]] = {
    "community_detection": {
        "metric": "test_best_rerun_accuracy",
        "label": "Community detection accuracy",
        "higher_is_better": True,
    },
    "triangle_counting": {
        "metric": "test_mse_by_total_triangles",
        "label": "Triangle MSE / total triangles",
        "higher_is_better": False,
    },
}
PAIR_KEYS: Final[tuple[str, ...]] = (
    "experiment",
    "train_seed",
    "homophily",
    "avg_degree",
    "power_law",
)


def load_and_validate_results(
    result_paths: dict[PolicyKey, Path],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load policy grids and verify the controlled-pairing contract.

    Each policy must contain 72 unique task/setting/seed rows. Enforcing
    that contract here prevents incomplete grids or duplicate runs from
    silently entering the paired statistical analysis.

    Parameters
    ----------
    result_paths : dict[PolicyKey, Path]
        Mapping from coordinate-policy key to its challenge JSON file.

    Returns
    -------
    tuple[pandas.DataFrame, pandas.DataFrame]
        Concatenated result rows and a compact validation table.

    Raises
    ------
    ValueError
        If a payload has an invalid schema, row count, or unique-key
        count.
    """
    frames: list[pd.DataFrame] = []
    validation_rows: list[dict[str, object]] = []

    for policy, result_path in result_paths.items():
        payload: dict[str, Any] = json.loads(
            result_path.read_text(encoding="utf-8")
        )
        rows = payload.get("results")
        metadata = payload.get("metadata")
        if not isinstance(rows, list) or not isinstance(metadata, dict):
            raise ValueError(f"Invalid results schema: {result_path}")

        frame = pd.DataFrame(rows)
        frame["policy"] = policy
        unique_keys = frame[list(PAIR_KEYS)].drop_duplicates().shape[0]
        declared_runs = metadata.get("n_runs")
        if len(rows) != 72 or declared_runs != 72 or unique_keys != 72:
            raise ValueError(
                f"Incomplete or duplicated result grid: {result_path}"
            )

        frames.append(frame)
        validation_rows.append(
            {
                "Policy": POLICY_LABELS[policy],
                "Model config": metadata.get("model_config"),
                "Study ID": metadata.get("study_id"),
                "Rows": len(rows),
                "Unique setting/seed/task keys": unique_keys,
            }
        )

    return (
        pd.concat(frames, ignore_index=True),
        pd.DataFrame(validation_rows),
    )


# Validate before computing any statistics or rendering any figures.
results, validation_table = load_and_validate_results(RESULT_PATHS)
display(validation_table)
print(
    "Validated two complete 72-run result grids from "
    "2026_tdl_challenge/outputs/."
)

,Policy,Model config,Study ID,Rows,Unique setting/seed/task keys
0,No coordinates,combinatorial/etnn_coordinate_policy_none,2026-07-14_01-10-33,72,72
1,Structural LapPE,combinatorial/etnn_coordinate_policy_lappe,2026-07-14_03-50-01,72,72


Validated two complete 72-run result grids from 2026_tdl_challenge/outputs/.


## In-Distribution GraphUniverse Comparison

Community detection uses test accuracy, where higher is better.
Triangle counting uses test MSE divided by the total structural triangle
count, where lower is better. The normalization is the challenge
notebook's reported metric, not an alternative loss introduced here.

Because both policies were evaluated over identical settings and seeds,
policy effects are analyzed as paired differences. Define the LapPE
advantage $\Delta$ so that positive is always favorable to LapPE:

$$
\Delta_{\mathrm{CD}} = \mathrm{Accuracy}_{\mathrm{LapPE}}
- \mathrm{Accuracy}_{\mathrm{none}},
$$

$$
\Delta_{\mathrm{TC}} = \mathrm{MSE}_{\mathrm{none}}
- \mathrm{MSE}_{\mathrm{LapPE}}.
$$

In [2]:
def paired_results_frame(frame: pd.DataFrame) -> pd.DataFrame:
    """Align both policies by task, graph regime, and training seed.

    The returned ``lappe_advantage`` column is sign-normalized so that a
    positive value always favors structural LapPE, even for triangle MSE
    where lower values are preferable.

    Parameters
    ----------
    frame : pandas.DataFrame
        Long-form result rows containing both coordinate policies.

    Returns
    -------
    pandas.DataFrame
        One paired row per task, setting, and seed.

    Raises
    ------
    ValueError
        If either policy is missing from a controlled pair.
    """
    parts: list[pd.DataFrame] = []
    for experiment, spec in EXPERIMENTS.items():
        selected = frame.loc[frame["experiment"] == experiment]
        wide = selected.pivot(
            index=list(PAIR_KEYS),
            columns="policy",
            values=spec["metric"],
        ).reset_index()
        if wide[["none", "lappe"]].isna().any().any():
            raise ValueError(f"Unpaired policy rows for {experiment}")

        wide["raw_delta_lappe_minus_none"] = wide["lappe"] - wide["none"]
        wide["lappe_advantage"] = (
            wide["raw_delta_lappe_minus_none"]
            if spec["higher_is_better"]
            else -wide["raw_delta_lappe_minus_none"]
        )
        parts.append(wide)
    return pd.concat(parts, ignore_index=True)


def bootstrap_mean_ci(
    values: NDArray[np.float64],
    *,
    seed: int = 20260714,
    draws: int = 20_000,
) -> tuple[float, float]:
    """Estimate a deterministic percentile interval for a mean.

    Parameters
    ----------
    values : numpy.typing.NDArray[numpy.float64]
        One-dimensional paired policy effects.
    seed : int, default=20260714
        Random seed used only for bootstrap resampling.
    draws : int, default=20000
        Number of bootstrap samples.

    Returns
    -------
    tuple[float, float]
        Lower and upper bounds of the two-sided 95% interval.

    Raises
    ------
    ValueError
        If the input is empty, not one-dimensional, or ``draws`` is not
        positive.
    """
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 1 or values.size == 0 or draws <= 0:
        raise ValueError("Bootstrap input must be a non-empty 1D array.")

    rng = np.random.default_rng(seed)
    indices = rng.integers(0, values.size, size=(draws, values.size))
    bootstrap_means = values[indices].mean(axis=1)
    quantiles = np.quantile(bootstrap_means, [0.025, 0.975])
    return float(quantiles[0]), float(quantiles[1])


def summarize_in_distribution(
    paired_frame: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, dict[str, Any]]]:
    """Summarize paired in-distribution effects for both tasks.

    Parameters
    ----------
    paired_frame : pandas.DataFrame
        Output of :func:`paired_results_frame`.

    Returns
    -------
    tuple[pandas.DataFrame, dict[str, dict[str, Any]]]
        Summary table and records keyed by experiment name.
    """
    records: list[dict[str, Any]] = []
    by_experiment: dict[str, dict[str, Any]] = {}

    for experiment, spec in EXPERIMENTS.items():
        selected = paired_frame.loc[paired_frame["experiment"] == experiment]
        advantage = selected["lappe_advantage"].to_numpy(dtype=np.float64)
        ci_low, ci_high = bootstrap_mean_ci(advantage)
        wilcoxon = stats.wilcoxon(advantage, zero_method="wilcox")
        record: dict[str, Any] = {
            "Task": spec["label"],
            "No coordinates (mean +/- std)": (
                f"{selected['none'].mean():.6f} +/- "
                f"{selected['none'].std(ddof=0):.6f}"
            ),
            "Structural LapPE (mean +/- std)": (
                f"{selected['lappe'].mean():.6f} +/- "
                f"{selected['lappe'].std(ddof=0):.6f}"
            ),
            "Raw LapPE - none": selected["raw_delta_lappe_minus_none"].mean(),
            "LapPE advantage": advantage.mean(),
            "Paired bootstrap 95% CI": (f"[{ci_low:+.6f}, {ci_high:+.6f}]"),
            "Pair wins (LapPE / none)": (
                f"{int((advantage > 0).sum())} / {int((advantage < 0).sum())}"
            ),
            "Wilcoxon p": wilcoxon.pvalue,
        }
        records.append(record)
        by_experiment[experiment] = record

    return pd.DataFrame(records), by_experiment


# Pair before summarizing so each policy comparison uses identical runs.
paired = paired_results_frame(results)
headline, summary_by_experiment = summarize_in_distribution(paired)
display(
    headline.style.format(
        {
            "Raw LapPE - none": "{:+.6f}",
            "LapPE advantage": "{:+.6f}",
            "Wilcoxon p": "{:.3f}",
        }
    )
)

,Task,No coordinates (mean +/- std),Structural LapPE (mean +/- std),Raw LapPE - none,LapPE advantage,Paired bootstrap 95% CI,Pair wins (LapPE / none),Wilcoxon p
0,Community detection accuracy,0.453502 +/- 0.129943,0.454346 +/- 0.131311,+0.000844,+0.000844,"[-0.000642, +0.002293]",23 / 13,0.315
1,Triangle MSE / total triangles,0.115299 +/- 0.100940,0.121076 +/- 0.104192,+0.005778,-0.005778,"[-0.026584, +0.013566]",16 / 20,0.346


In [3]:
def build_homophily_table(
    result_frame: pd.DataFrame,
    paired_frame: pd.DataFrame,
) -> pd.DataFrame:
    """Aggregate policy metrics within each homophily regime.

    Parameters
    ----------
    result_frame : pandas.DataFrame
        Long-form rows for both policies and tasks.
    paired_frame : pandas.DataFrame
        Paired effects produced by :func:`paired_results_frame`.

    Returns
    -------
    pandas.DataFrame
        Six rows: three homophily regimes per task.
    """
    rows: list[dict[str, object]] = []

    for experiment, spec in EXPERIMENTS.items():
        selected = result_frame.loc[result_frame["experiment"] == experiment]
        for homophily in HOMOPHILY_ORDER:
            row: dict[str, object] = {
                "Task": spec["label"],
                "Homophily": homophily.replace("h_", "").title(),
            }
            for policy in ("none", "lappe"):
                values = selected.loc[
                    (selected["policy"] == policy)
                    & (selected["homophily"] == homophily),
                    spec["metric"],
                ].to_numpy(dtype=np.float64)
                row[POLICY_LABELS[policy]] = (
                    f"{values.mean():.6f} +/- {values.std(ddof=0):.6f}"
                )

            pair_values = paired_frame.loc[
                (paired_frame["experiment"] == experiment)
                & (paired_frame["homophily"] == homophily),
                "lappe_advantage",
            ].to_numpy(dtype=np.float64)
            row["LapPE advantage"] = pair_values.mean()
            rows.append(row)

    return pd.DataFrame(rows)


homophily_table = build_homophily_table(results, paired)
display(homophily_table.style.format({"LapPE advantage": "{:+.6f}"}))

,Task,Homophily,No coordinates,Structural LapPE,LapPE advantage
0,Community detection accuracy,Lo,0.320205 +/- 0.004515,0.319982 +/- 0.005724,-0.000223
1,Community detection accuracy,Mid,0.420480 +/- 0.030158,0.420983 +/- 0.028766,+0.000503
2,Community detection accuracy,Hi,0.619821 +/- 0.056613,0.622071 +/- 0.059742,+0.002251
3,Triangle MSE / total triangles,Lo,0.024824 +/- 0.026626,0.025014 +/- 0.027290,-0.000189
4,Triangle MSE / total triangles,Mid,0.113449 +/- 0.033492,0.129072 +/- 0.075776,-0.015623
5,Triangle MSE / total triangles,Hi,0.207623 +/- 0.109651,0.209144 +/- 0.095045,-0.001521


In [4]:
cd: dict[str, Any] = summary_by_experiment["community_detection"]
tc: dict[str, Any] = summary_by_experiment["triangle_counting"]

# Derive the prose from computed statistics so the narrative cannot drift
# from the displayed result table after a future rerun.
interpretation = f"""**Controlled in-distribution result.** Structural
LapPE changes mean community accuracy by
`{cd["Raw LapPE - none"]:+.6f}` and mean normalized triangle MSE by
`{tc["Raw LapPE - none"]:+.6f}`. Both paired bootstrap intervals contain
zero, and the Wilcoxon p-values are `{cd["Wilcoxon p"]:.3f}` and
`{tc["Wilcoxon p"]:.3f}`. The experiment therefore does not establish a
universal performance winner. LapPE trends higher for community
detection, while `none` has lower mean triangle error.

This supports the coordinate-policy abstraction: structural geometry is
an explicit modeling assumption whose usefulness varies by task and
graph regime, not a feature that should be silently invented for every
coordinate-free graph."""
display(Markdown(interpretation))

**Controlled in-distribution result.** Structural
LapPE changes mean community accuracy by
`+0.000844` and mean normalized triangle MSE by
`+0.005778`. Both paired bootstrap intervals contain
zero, and the Wilcoxon p-values are `0.315` and
`0.346`. The experiment therefore does not establish a
universal performance winner. LapPE trends higher for community
detection, while `none` has lower mean triangle error.

This supports the coordinate-policy abstraction: structural geometry is
an explicit modeling assumption whose usefulness varies by task and
graph regime, not a feature that should be silently invented for every
coordinate-free graph.

## GraphUniverse Figures

In [5]:
def homophily_profile(
    frame: pd.DataFrame,
    *,
    experiment: str,
    policy: PolicyKey,
    metric: str,
) -> tuple[list[float], list[float]]:
    """Compute means and normal-approximation 95% confidence errors.

    Parameters
    ----------
    frame : pandas.DataFrame
        Long-form result rows.
    experiment : str
        GraphUniverse task identifier.
    policy : PolicyKey
        Coordinate policy to summarize.
    metric : str
        Result column used for the task.

    Returns
    -------
    tuple[list[float], list[float]]
        Means and 95% error-bar half-widths in low/mid/high order.
    """
    means: list[float] = []
    errors: list[float] = []
    for homophily in HOMOPHILY_ORDER:
        values = frame.loc[
            (frame["experiment"] == experiment)
            & (frame["policy"] == policy)
            & (frame["homophily"] == homophily),
            metric,
        ].to_numpy(dtype=np.float64)
        means.append(float(values.mean()))
        standard_error = values.std(ddof=1) / np.sqrt(values.size)
        errors.append(float(1.96 * standard_error))
    return means, errors


fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
x = np.arange(len(HOMOPHILY_ORDER))
for axis, (experiment, spec) in zip(axes, EXPERIMENTS.items(), strict=True):
    # Small horizontal offsets keep overlapping confidence intervals legible.
    for offset, policy in zip(
        (-0.06, 0.06),
        ("none", "lappe"),
        strict=True,
    ):
        means, errors = homophily_profile(
            results,
            experiment=experiment,
            policy=policy,
            metric=spec["metric"],
        )
        axis.errorbar(
            x + offset,
            means,
            yerr=errors,
            marker="o",
            markersize=6,
            linewidth=2,
            capsize=4,
            color=POLICY_COLORS[policy],
            label=POLICY_LABELS[policy],
        )
    axis.set_title(spec["label"], fontsize=11, fontweight="bold")
    axis.set_xticks(x, ["Low", "Mid", "High"])
    axis.set_xlabel("GraphUniverse homophily regime")
    axis.grid(axis="y", color="#D8D8D8", linewidth=0.8, alpha=0.8)
axes[0].set_ylabel("Accuracy (higher is better)")
axes[1].set_ylabel("MSE / total triangles (lower is better)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.9),
    ncol=2,
    frameon=False,
)
fig.suptitle(
    "ETNN coordinate policies across GraphUniverse regimes",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)
fig.tight_layout(rect=(0, 0, 1, 0.82), w_pad=2.2)

# Save the same deterministic figure displayed in the notebook for the PR.
profile_path = ASSET_DIR / "graphuniverse_homophily_profiles.png"
fig.savefig(profile_path, bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"Saved {profile_path.relative_to(ROOT)}")

Saved 2026_tdl_challenge/submissions/assets/etnn_coordinate_policy/graphuniverse_homophily_profiles.png


![Line charts comparing no-coordinate and structural-LapPE ETNN metrics
across low, mid, and high homophily.](assets/etnn_coordinate_policy/graphuniverse_homophily_profiles.png)

In [6]:
def policy_delta_matrix(
    paired_frame: pd.DataFrame,
    *,
    experiment: str,
    column_pairs: list[tuple[str, str]],
) -> NDArray[np.float64]:
    """Average paired LapPE advantages over seeds for one task.

    Parameters
    ----------
    paired_frame : pandas.DataFrame
        Paired policy effects for all tasks and regimes.
    experiment : str
        Task identifier to place in one heatmap panel.
    column_pairs : list[tuple[str, str]]
        Ordered degree and power-law combinations.

    Returns
    -------
    numpy.typing.NDArray[numpy.float64]
        Matrix indexed by homophily rows and regime-pair columns.
    """
    selected = paired_frame.loc[paired_frame["experiment"] == experiment]
    matrix = np.empty(
        (len(HOMOPHILY_ORDER), len(column_pairs)),
        dtype=np.float64,
    )
    for row_index, homophily in enumerate(HOMOPHILY_ORDER):
        for column_index, (degree, power_law) in enumerate(column_pairs):
            values = selected.loc[
                (selected["homophily"] == homophily)
                & (selected["avg_degree"] == degree)
                & (selected["power_law"] == power_law),
                "lappe_advantage",
            ]
            matrix[row_index, column_index] = values.mean()
    return matrix


column_pairs: list[tuple[str, str]] = [
    (degree, power_law)
    for degree in DEGREE_ORDER
    for power_law in POWER_LAW_ORDER
]
column_labels: list[str] = [
    f"{degree.replace('d_', '').title()} degree\n"
    f"{power_law.replace('pl_', '').title()} power law"
    for degree, power_law in column_pairs
]
matrices: list[NDArray[np.float64]] = [
    policy_delta_matrix(
        paired,
        experiment=experiment,
        column_pairs=column_pairs,
    )
    for experiment in EXPERIMENTS
]

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), constrained_layout=True)
for axis, (_experiment, spec), matrix in zip(
    axes, EXPERIMENTS.items(), matrices, strict=True
):
    # Use a task-specific symmetric scale because the task metrics have
    # different magnitudes while zero has the same policy meaning.
    max_abs = float(np.abs(matrix).max())
    image = axis.imshow(
        matrix,
        # Blue marks negative values (none better); red marks positive
        # values (LapPE better), matching the signed advantage scale.
        cmap="RdBu_r",
        vmin=-max_abs,
        vmax=max_abs,
        aspect="auto",
    )
    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            value = matrix[row_index, column_index]
            axis.text(
                column_index,
                row_index,
                f"{value:+.4f}",
                ha="center",
                va="center",
                fontsize=9,
                color="white" if abs(value) > 0.55 * max_abs else "#202020",
            )
    axis.set_title(spec["label"], fontsize=11, fontweight="bold")
    axis.set_xticks(np.arange(len(column_pairs)), column_labels, fontsize=8)
    axis.set_yticks(np.arange(len(HOMOPHILY_ORDER)), ["Low", "Mid", "High"])
    axis.set_xlabel("Degree and power-law regime")
    axis.set_ylabel("Homophily regime")
    colorbar = fig.colorbar(image, ax=axis, shrink=0.82, pad=0.02)
    colorbar.set_label("LapPE advantage (blue: none; red: LapPE)")
fig.suptitle(
    "Paired policy deltas averaged over three seeds",
    fontsize=14,
    fontweight="bold",
)
delta_path = ASSET_DIR / "graphuniverse_paired_delta_heatmaps.png"
fig.savefig(delta_path, bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"Saved {delta_path.relative_to(ROOT)}")

Saved 2026_tdl_challenge/submissions/assets/etnn_coordinate_policy/graphuniverse_paired_delta_heatmaps.png


Blue indicates a negative LapPE advantage (`none` performs better); red
indicates a positive LapPE advantage (structural LapPE performs better).

![Two heatmaps of paired structural-LapPE advantages across homophily,
degree, and power-law regimes.](assets/etnn_coordinate_policy/graphuniverse_paired_delta_heatmaps.png)

In [7]:
def build_tradeoff_frame(paired_frame: pd.DataFrame) -> pd.DataFrame:
    """Place both task advantages on one row per setting and seed.

    Parameters
    ----------
    paired_frame : pandas.DataFrame
        Paired coordinate-policy effects for both tasks.

    Returns
    -------
    pandas.DataFrame
        Wide frame with community and triangle advantages as columns.
    """
    return paired_frame.pivot(
        index=["train_seed", "homophily", "avg_degree", "power_law"],
        columns="experiment",
        values="lappe_advantage",
    ).reset_index()


tradeoff = build_tradeoff_frame(paired)
homophily_colors: dict[str, str] = {
    "h_lo": "#3B82A0",
    "h_mid": "#D18B2C",
    "h_hi": "#8A5A9E",
}
fig, axis = plt.subplots(figsize=(7.2, 6.1), constrained_layout=True)
for homophily in HOMOPHILY_ORDER:
    selected = tradeoff.loc[tradeoff["homophily"] == homophily]
    axis.scatter(
        selected["community_detection"],
        selected["triangle_counting"],
        s=58,
        alpha=0.82,
        color=homophily_colors[homophily],
        edgecolor="white",
        linewidth=0.7,
        label=homophily.replace("h_", "").title(),
    )

# The zero lines create four quadrants with an immediate policy meaning.
axis.axhline(0, color="#555555", linewidth=1)
axis.axvline(0, color="#555555", linewidth=1)
axis.grid(color="#E0E0E0", linewidth=0.8, alpha=0.75)
axis.set_xlabel("LapPE advantage: community accuracy")
axis.set_ylabel("LapPE advantage: triangle error reduction")
axis.set_title(
    "Structural-coordinate tradeoff by setting and seed",
    fontsize=13,
    fontweight="bold",
)
axis.text(
    0.98,
    0.98,
    "LapPE improves both",
    transform=axis.transAxes,
    ha="right",
    va="top",
    fontsize=9,
    color="#3D6A4E",
)
axis.text(
    0.02,
    0.03,
    "No coordinates improve both",
    transform=axis.transAxes,
    ha="left",
    va="bottom",
    fontsize=9,
    color="#6B4A3A",
)
axis.legend(title="Homophily", frameon=False)
tradeoff_path = ASSET_DIR / "graphuniverse_task_tradeoff.png"
fig.savefig(tradeoff_path, bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"Saved {tradeoff_path.relative_to(ROOT)}")

Saved 2026_tdl_challenge/submissions/assets/etnn_coordinate_policy/graphuniverse_task_tradeoff.png


![Scatter plot comparing LapPE advantages for community detection and
triangle counting by setting and seed.](assets/etnn_coordinate_policy/graphuniverse_task_tradeoff.png)

## Out-of-Distribution Transfer

Each trained model is also evaluated on the other 11 GraphUniverse
settings. Those evaluations are correlated within a trained model, so
the analysis first averages the 11 OOD values for each of the 36
task-specific training runs. Confidence intervals are then bootstrapped
over those 36 training-run clusters rather than treating all 396
train/evaluation pairs as independent observations.

In [8]:
def ood_training_run_means(frame: pd.DataFrame) -> pd.DataFrame:
    """Average the 11 OOD evaluations within each trained model.

    Averaging at the training-run level preserves the correct clustering
    unit: the 11 OOD evaluations from one fitted model are correlated and
    must not be treated as independent observations.

    Parameters
    ----------
    frame : pandas.DataFrame
        Long-form policy rows with an ``ood_test`` mapping per run.

    Returns
    -------
    pandas.DataFrame
        One OOD mean per policy, task, setting, and training seed.

    Raises
    ------
    ValueError
        If a row lacks the expected 11 finite OOD task metrics.
    """
    records: list[dict[str, object]] = []
    for _, row in frame.iterrows():
        experiment = str(row["experiment"])
        spec = EXPERIMENTS[experiment]
        metric = spec["metric"]
        ood_test = row["ood_test"]
        if not isinstance(ood_test, dict):
            raise ValueError("Expected an OOD mapping for every run.")

        values: list[float] = []
        for item in ood_test.values():
            if not isinstance(item, dict):
                raise ValueError("Expected OOD entries to be mappings.")
            metric_value = item.get(metric)
            if metric_value is None:
                continue
            numeric_value = float(metric_value)
            if math.isfinite(numeric_value):
                values.append(numeric_value)

        if len(values) != 11:
            raise ValueError(
                f"Expected 11 finite OOD values, found {len(values)}."
            )
        records.append(
            {
                "policy": row["policy"],
                "experiment": experiment,
                "train_seed": row["train_seed"],
                "homophily": row["homophily"],
                "avg_degree": row["avg_degree"],
                "power_law": row["power_law"],
                "ood_mean": float(np.mean(values)),
            }
        )
    return pd.DataFrame(records)


def summarize_ood(ood_frame: pd.DataFrame) -> pd.DataFrame:
    """Create clustered OOD policy summaries for both tasks.

    Parameters
    ----------
    ood_frame : pandas.DataFrame
        Training-run means returned by :func:`ood_training_run_means`.

    Returns
    -------
    pandas.DataFrame
        Task-level means, paired effects, uncertainty, wins, and tests.
    """
    records: list[dict[str, object]] = []
    index = list(PAIR_KEYS)

    for experiment, spec in EXPERIMENTS.items():
        selected = ood_frame.loc[ood_frame["experiment"] == experiment]
        wide = selected.pivot(
            index=index,
            columns="policy",
            values="ood_mean",
        ).reset_index()
        advantage = (
            wide["lappe"] - wide["none"]
            if spec["higher_is_better"]
            else wide["none"] - wide["lappe"]
        ).to_numpy(dtype=np.float64)
        ci_low, ci_high = bootstrap_mean_ci(advantage)
        records.append(
            {
                "Task": spec["label"],
                "No coordinates OOD mean": wide["none"].mean(),
                "Structural LapPE OOD mean": wide["lappe"].mean(),
                "LapPE advantage": advantage.mean(),
                "Cluster bootstrap 95% CI": (
                    f"[{ci_low:+.6f}, {ci_high:+.6f}]"
                ),
                "Training-run wins (LapPE / none)": (
                    f"{int((advantage > 0).sum())} / "
                    f"{int((advantage < 0).sum())}"
                ),
                "Wilcoxon p": stats.wilcoxon(advantage).pvalue,
            }
        )

    return pd.DataFrame(records)


# Bootstrap over 36 fitted models per task, not 396 correlated OOD rows.
ood_runs = ood_training_run_means(results)
ood_summary = summarize_ood(ood_runs)
display(
    ood_summary.style.format(
        {
            "No coordinates OOD mean": "{:.6f}",
            "Structural LapPE OOD mean": "{:.6f}",
            "LapPE advantage": "{:+.6f}",
            "Wilcoxon p": "{:.3f}",
        }
    )
)

,Task,No coordinates OOD mean,Structural LapPE OOD mean,LapPE advantage,Cluster bootstrap 95% CI,Training-run wins (LapPE / none),Wilcoxon p
0,Community detection accuracy,0.366667,0.367668,+0.001001,"[-0.001157, +0.003101]",23 / 13,0.162
1,Triangle MSE / total triangles,2.866467,2.715533,+0.150934,"[-1.148205, +1.266967]",19 / 17,0.371


The clustered OOD intervals also cross zero. OOD transfer therefore
reinforces the in-distribution conclusion: policy effects are modest
relative to GraphUniverse regime effects, and neither coordinate
assumption is uniformly superior.

## QM9 Physical-Coordinate Validation

GraphUniverse cannot validate the physical ETNN path, so QM9 is used for
three complementary checks:

1. a paper-faithful native NSAPH reference;
2. a controlled port that substitutes only the architecture-matched
   TopoBench ETNN core;
3. a reduced end-to-end integration study of the submitted physical policy.

### Paper-faithful native reference and controlled TopoBench port

The pinned official NSAPH implementation uses `experiment_1`, the `egnn`
split, seed 42, width 128, seven ETNN layers, batch size 96, learning rate
`5e-4`, weight decay `1e-5`, and a 1,000-epoch schedule. This corresponds
to Table 11, configuration 1, `[A+B|max|0|1|1]`, in the ETNN paper.

The controlled TopoBench run fixes the processed QM9CC molecules, ordered
split, 15-channel atom and 19-channel bond features, physical coordinates,
five-channel geometric invariants, target normalization, complete seed-42
initial state, optimizer, schedule, and molecular readout. The only intended
model substitution is the TopoBench ETNN message/update core. The included
QM9 adapter and parity wrapper make this boundary explicit.

Real-batch parity checks match discrete structure exactly and match model
outputs, loss, BatchNorm buffers, and gradients to floating-point tolerance.
After one Adam step, four parameter tensors differ by at most `2.17e-6`;
the port therefore establishes numerical rather than bitwise optimizer
parity.


In [9]:
class QM9ResultRecord(TypedDict):
    """Represent one public QM9 comparison result."""

    implementation: str
    best_validation_epoch: int | None
    best_validation_mae: float | None
    test_mae: float
    value_precision: str


def load_and_validate_qm9_comparison(path: Path) -> dict[str, Any]:
    """Load the public QM9 comparison and enforce its protocol contract.

    Parameters
    ----------
    path : pathlib.Path
        JSON artifact containing paper, native, and TopoBench results.

    Returns
    -------
    dict[str, Any]
        Validated comparison payload.

    Raises
    ------
    ValueError
        If the benchmark, protocol, result labels, or metrics differ from
        the audited controlled comparison.
    """
    payload: dict[str, Any] = json.loads(path.read_text(encoding="utf-8"))
    if payload.get("benchmark") != "QM9" or payload.get("target") != "mu":
        raise ValueError("Expected the QM9 dipole-moment comparison.")

    protocol = payload.get("protocol")
    expected_protocol: dict[str, object] = {
        "batch_size": 96,
        "epochs": 1000,
        "hidden_channels": 128,
        "learning_rate": 5e-4,
        "num_layers": 7,
        "scheduler_t_max": 333,
        "seed": 42,
        "split": "egnn",
        "weight_decay": 1e-5,
    }
    if not isinstance(protocol, dict):
        raise ValueError("QM9 protocol must be a mapping.")
    for key, expected_value in expected_protocol.items():
        if protocol.get(key) != expected_value:
            raise ValueError(f"Unexpected QM9 protocol value for {key}.")

    records = payload.get("results")
    if not isinstance(records, list) or len(records) != 3:
        raise ValueError("Expected paper, native, and TopoBench QM9 rows.")
    expected_labels = {
        "ETNN paper, Table 11, configuration 1",
        "Native NSAPH reproduction",
        "Protocol-matched TopoBench ETNN port",
    }
    observed_labels = {
        str(record.get("implementation"))
        for record in records
        if isinstance(record, dict)
    }
    if observed_labels != expected_labels:
        raise ValueError("QM9 implementation labels do not match the audit.")
    if any(
        not math.isfinite(float(record["test_mae"]))
        for record in records
        if isinstance(record, dict)
    ):
        raise ValueError("QM9 test metrics must be finite.")
    return payload


QM9_COMPARISON_PATH: Final[Path] = (
    ROOT / "2026_tdl_challenge/submissions/assets/etnn_coordinate_policy/"
    "qm9_protocol_matched_comparison.json"
)
qm9_comparison = load_and_validate_qm9_comparison(QM9_COMPARISON_PATH)
qm9_controlled_records: list[QM9ResultRecord] = qm9_comparison["results"]

qm9_controlled_table = pd.DataFrame(
    [
        {
            "Reference": record["implementation"],
            "Best validation MAE (D)": record["best_validation_mae"],
            "Selected epoch": record["best_validation_epoch"],
            "Test MAE (D)": record["test_mae"],
        }
        for record in qm9_controlled_records
    ]
)
display(
    qm9_controlled_table.style.format(
        {
            "Best validation MAE (D)": lambda value: (
                "Not reported" if pd.isna(value) else f"{value:.6f}"
            ),
            "Selected epoch": lambda value: (
                "Not reported" if pd.isna(value) else f"{int(value)}"
            ),
            "Test MAE (D)": "{:.6f}",
        }
    )
)

paper_test_mae = float(qm9_controlled_records[0]["test_mae"])
native_test_mae = float(qm9_controlled_records[1]["test_mae"])
topobench_test_mae = float(qm9_controlled_records[2]["test_mae"])
test_delta = topobench_test_mae - native_test_mae
relative_delta = 100.0 * test_delta / native_test_mae

fig, axis = plt.subplots(figsize=(8.2, 3.8), constrained_layout=True)
labels = ["Native NSAPH", "TopoBench ETNN port"]
values = [native_test_mae, topobench_test_mae]
colors = ["#2F5D62", "#C65D3B"]
bars = axis.barh(labels, values, color=colors, height=0.52)
axis.axvline(
    paper_test_mae,
    color="#555555",
    linestyle="--",
    linewidth=1.5,
    label="Paper result (rounded)",
)
for bar, value in zip(bars, values, strict=True):
    axis.text(
        value + 0.00025,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.6f} D",
        va="center",
        fontsize=10,
    )
axis.set_xlim(0.0, 0.0355)
axis.invert_yaxis()
axis.set_xlabel("QM9 dipole-moment test MAE (lower is better)")
axis.set_title(
    "Paper-aligned ETNN validation on QM9",
    fontsize=13,
    fontweight="bold",
)
axis.grid(axis="x", color="#E0E0E0", linewidth=0.8, alpha=0.75)
axis.legend(frameon=False, loc="center right")
qm9_comparison_path = ASSET_DIR / "qm9_protocol_matched_comparison.png"
fig.savefig(qm9_comparison_path, bbox_inches="tight", facecolor="white")
plt.close(fig)

print(
    "TopoBench - native test MAE: "
    f"{test_delta:+.6f} D ({relative_delta:+.2f}%)."
)
print(f"Saved {qm9_comparison_path.relative_to(ROOT)}")

,Reference,Best validation MAE (D),Selected epoch,Test MAE (D)
0,"ETNN paper, Table 11, configuration 1",Not reported,Not reported,0.030000
1,Native NSAPH reproduction,0.030985,1000,0.031815
2,Protocol-matched TopoBench ETNN port,0.030401,999,0.031217


TopoBench - native test MAE: -0.000598 D (-1.88%).
Saved 2026_tdl_challenge/submissions/assets/etnn_coordinate_policy/qm9_protocol_matched_comparison.png


![Horizontal bars comparing native NSAPH and the protocol-matched
TopoBench ETNN port against the rounded paper result.](assets/etnn_coordinate_policy/qm9_protocol_matched_comparison.png)

The protocol-matched TopoBench port reaches `0.031217 D` test MAE versus
`0.031815 D` for the native reproduction, a difference of `-0.000598 D`
(`-1.88%`). This single-seed controlled result supports training-level
fidelity of the TopoBench ETNN core. It does not establish general
superiority or independently validate a TopoBench molecular lifting because
both runs consume the same native QM9CC batches and invariants. The paper's
`0.030 D` value is rounded to three decimal places.

### Submitted TopoBench physical-policy integration

A separate reduced comparison verifies that atom positions survive generic
TopoBench lifting and that physical invariants, message passing, optional
coordinate updates, checkpoint restoration, and test evaluation run end to
end.

**Protocol:** target index 0 (dipole moment $\mu$, Debye), seed 42,
batch size 128, 10 epochs, 50 train batches per epoch, 10 validation
batches, 10 test batches, and invariant BatchNorm. Each configuration
receives 500 optimizer steps.

This integration study uses a small width-32/two-layer model, a TopoBench
split, generic triangle-induced lifting, and no paper-specific molecular
virtual cell, rings, or functional-group construction. Its metrics should
not be compared directly with the controlled result above.


In [10]:
# These fixed records were obtained through the controlled QM9
# validation procedure described above. They are reported separately
# because QM9 is an integration benchmark, not a challenge JSON.
qm9_records: list[dict[str, object]] = [
    {
        "Configuration": "Physical static",
        "Position update": False,
        "Hausdorff": True,
        "Best validation MAE": 0.808063,
        "Test MAE": 0.818540,
        "Test MSE": 1.163962,
        "Approximate runtime": "58 min incl. preprocessing",
    },
    {
        "Configuration": "Dynamic, no Hausdorff",
        "Position update": True,
        "Hausdorff": False,
        "Best validation MAE": 0.865539,
        "Test MAE": 0.873744,
        "Test MSE": 1.277022,
        "Approximate runtime": "12 min",
    },
    {
        "Configuration": "Full physical",
        "Position update": True,
        "Hausdorff": True,
        "Best validation MAE": 0.861778,
        "Test MAE": 0.868965,
        "Test MSE": 1.275474,
        "Approximate runtime": "99 min",
    },
]
qm9 = pd.DataFrame(qm9_records)
display(
    qm9.style.format(
        {
            "Best validation MAE": "{:.6f}",
            "Test MAE": "{:.6f}",
            "Test MSE": "{:.6f}",
        }
    )
)

,Configuration,Position update,Hausdorff,Best validation MAE,Test MAE,Test MSE,Approximate runtime
0,Physical static,False,True,0.808063,0.818540,1.163962,58 min incl. preprocessing
1,"Dynamic, no Hausdorff",True,False,0.865539,0.873744,1.277022,12 min
2,Full physical,True,True,0.861778,0.868965,1.275474,99 min


Static physical invariants perform best in this short regime. Learned
coordinate updates do not improve the reduced metric, and Hausdorff
channels recover only a small amount relative to dynamic-no-Hausdorff
while increasing runtime substantially. This does not invalidate the
equivariant update; it shows that a faithful dynamic path is not
automatically advantageous under a short, generic molecular protocol.

## Faithfulness and Deviations

| Policy | Faithful ETNN elements | TopoBench adaptation / limitation |
|---|---|---|
| `none` | Typed cell relations, relation-specific gated messages, rank-wise residual feature updates | Omits geometric invariants and coordinate updates because coordinates are absent. |
| `structural_lappe` | Adds an invariant distance to typed ETNN messages | LapPE is graph-derived rather than physical; higher-rank coordinates use recursive incidence averaging; coordinates remain fixed. |
| `physical` | Physical cell invariants, NSAPH-style gated messages, optional radial rank-0 coordinate update, invariant recomputation between layers | Vertex memberships are reconstructed from TopoBench incidence tensors; dense membership and Hausdorff calculations target small/medium physical datasets. |

The consolidated implementation never silently interprets an arbitrary
`pos` attribute or structural embedding as physical geometry. Policy
selection is explicit in Hydra configuration, and missing required
coordinates raise clear errors.

## Conclusions

1. **Challenge compatibility:** `none` and `structural_lappe` each
   complete the full 72-run GraphUniverse grid and produce distinct
   result artifacts.
2. **No universal GraphUniverse winner:** paired in-distribution and
   clustered OOD intervals cross zero. LapPE trends toward better
   community detection; `none` has lower mean triangle error.
3. **Native reference fidelity:** the paper-faithful native QM9
   reproduction obtains `0.031815 D` test MAE versus the paper's rounded
   `0.030 D`.
4. **Controlled TopoBench core fidelity:** with native QM9CC inputs and
   protocol held fixed, the architecture-matched TopoBench ETNN port
   obtains `0.031217 D` test MAE. This is a controlled single-seed port
   validation, not an independent molecular-lifting benchmark.
5. **Physical ETNN is functional end to end:** reduced QM9 runs exercise
   physical invariants and optional E(n)-equivariant coordinate dynamics
   through the submitted TopoBench pipeline.
6. **Semantics over silent fallback:** the coordinate policy makes the
   dataset's geometric assumptions explicit and keeps structural
   pseudo-coordinates distinct from physical positions.

The official GraphUniverse notebook selects
`combinatorial/etnn_coordinate_policy_lappe` to exercise the
structural-coordinate path. The paired `none` result is committed as a
controlled baseline. Physical mode is documented through QM9 rather
than an invalid GraphUniverse JSON.


## Reproducibility and Artifacts

**GraphUniverse validation:** two complete 72-run grids over identical
settings and seeds.

**Native QM9 reference:** one paper-faithful, single-seed run of the
pinned official NSAPH implementation.

**Protocol-matched TopoBench QM9 validation:** one single-seed controlled
port run using the same native QM9CC batches, invariants, initialization,
and outer training protocol. The public adapter and parity wrapper expose
the exact model-core substitution.

**Reduced TopoBench QM9 validation:** three controlled physical-coordinate
configurations, each completing training, checkpoint restoration,
validation, and testing through the submitted generic pipeline.

```text
2026_tdl_challenge/outputs/etnn_coordinate_policy_none/results.json
2026_tdl_challenge/outputs/etnn_coordinate_policy_lappe/results.json
2026_tdl_challenge/submissions/assets/etnn_coordinate_policy/qm9_protocol_matched_comparison.json
2026_tdl_challenge/submissions/assets/etnn_coordinate_policy/
```

Execute this notebook from the repository root:

```bash
MPLCONFIGDIR=/private/tmp .venv/bin/jupyter nbconvert \
  --to notebook --execute --inplace \
  --ExecutePreprocessor.timeout=600 \
  2026_tdl_challenge/submissions/etnn_coordinate_policy_comparison.ipynb
```

All displayed tables and figures are regenerated without network access
from the committed JSON artifacts.
